# Genéricos Avanzados, Reflexión y Testing en Go

Este notebook profundiza en características de Go que permiten escribir código altamente reutilizable y dinámico, además de las herramientas integradas para asegurar la calidad del código.

## 1. Genéricos Avanzados (Go 1.18+)

Los genéricos permiten parametrizar funciones y tipos. Se basan en **Type Sets** (conjuntos de tipos) definidos a través de interfaces.

In [ ]:
package main

import "fmt"

// Definición de un Constraint (Restricción)
// El símbolo ~ permite incluir tipos cuya base sea int (ej. type MiInt int)
type Numero interface {
    ~int | ~int64 | ~float64
}

// Estructura Genérica
type Contenedor[T any] struct {
    Valor T
}

// Función Genérica con Constraint
func Sumar[T Numero](a, b T) T {
    return a + b
}

func main() {
    fmt.Println(Sumar(10, 20))
    fmt.Println(Sumar(1.5, 2.5))
}

## 2. Reflexión (`reflect`)

La reflexión permite inspeccionar tipos y valores en tiempo de ejecución. Es potente pero debe usarse con moderación por su costo en rendimiento y pérdida de seguridad de tipos estática.

In [ ]:
import "reflect"

func inspeccionar(obj any) {
    t := reflect.TypeOf(obj)
    v := reflect.ValueOf(obj)

    fmt.Printf("Tipo: %v, Kind: %v\n", t, t.Kind())

    if t.Kind() == reflect.Struct {
        for i := 0; i < t.NumField(); i++ {
            campo := t.Field(i)
            valor := v.Field(i)
            fmt.Printf("Campo: %s, Valor: %v\n", campo.Name, valor)
        }
    }
}

type User struct {
    ID   int
    Name string
}

// inspeccionar(User{1, "Maxi"})

## 3. Estructuras de Datos Propias de Go

### Los Slices bajo el capó
Un slice es un descriptor que contiene:
1. Puntero al array subyacente.
2. Longitud (`len`).
3. Capacidad (`cap`).

In [ ]:
s := make([]int, 0, 5)
fmt.Printf("Len: %d, Cap: %d\n", len(s), cap(s))

s = append(s, 1, 2, 3)
// Si append supera la capacidad, Go reserva un nuevo array mayor y copia los datos.

## 4. Testing en Go

Go incluye una herramienta de testing integrada muy potente. Los archivos de test terminan en `_test.go`.

In [ ]:
// Ejemplo de estructura de un test (normalmente en un archivo .go aparte)
/*
import "testing"

func TestSumar(t *testing.T) {
    resultado := Sumar(2, 3)
    esperado := 5
    if resultado != esperado {
        t.Errorf("Resultado incorrecto, esperado %d, obtenido %d", esperado, resultado)
    }
}

// Table Driven Tests (Patrón recomendado en Go)
func TestSumarTable(t *testing.T) {
    casos := []struct {
        a, b, esperado int
    }{
        {1, 1, 2},
        {2, 2, 4},
        {-1, 1, 0},
    }

    for _, c := range casos {
        res := Sumar(c.a, c.b)
        if res != c.esperado {
            t.Errorf("Sumar(%d, %d) == %d, esperado %d", c.a, c.b, res, c.esperado)
        }
    }
}
*/

## 5. Benchmarks

Permiten medir el rendimiento de tu código.

In [ ]:
/*
func BenchmarkSumar(b *testing.B) {
    for i := 0; i < b.N; i++ {
        Sumar(10, 20)
    }
}
*/

// Ejecutar con: go test -bench=.

## 6. Fuzzing (Go 1.18+)

Técnica de testing automatizado que genera entradas aleatorias para encontrar fallos.

In [ ]:
/*
func FuzzSumar(f *testing.F) {
    f.Add(10, 20)
    f.Fuzz(func(t *testing.T, a int, b int) {
        Sumar(a, b)
    })
}
*/

// Ejecutar con: go test -fuzz=Fuzz